In [1]:
# IMPORTS FROM PYOMO

from pyomo.environ import (
    Block,
    ConcreteModel,
    Var,
    Param,
    Constraint,
    Objective,
    Expression,
    value,
    check_optimal_termination,
    assert_optimal_termination,
    TransformationFactory,
    units as pyunits,
)

from pyomo.util.check_units import assert_units_consistent
from pyomo.network import Arc


# IMPORTS FROM IDAES
from idaes.core import FlowsheetBlock, UnitModelCostingBlock
from idaes.models.unit_models import Feed, Product, StateJunction
from idaes.core.util.model_statistics import degrees_of_freedom
from idaes.core.util.scaling import calculate_scaling_factors, set_scaling_factor, constraint_scaling_transform
from idaes.core.util.initialization import propagate_state


# IMPORTS FROM WaterTAP
from watertap.property_models.NaCl_prop_pack import NaClParameterBlock
from watertap.property_models.seawater_prop_pack import SeawaterParameterBlock
from watertap.unit_models.pressure_changer import Pump
from watertap.unit_models.reverse_osmosis_0D import (
    ReverseOsmosis0D,
    ConcentrationPolarizationType,
    MassTransferCoefficient,
    PressureChangeType,
)
from watertap.unit_models.reverse_osmosis_1D import (
    ReverseOsmosis1D,
    PressureChangeType,
    MassTransferCoefficient,
    ConcentrationPolarizationType,
)

from watertap.core.solvers import get_solver

# TRANSLATOR FUNCTION
import idaes.logger as idaeslog
from idaes.core import declare_process_block_class
from idaes.core.util.exceptions import InitializationError
from idaes.models.unit_models.translator import TranslatorData

from watertap.core.util.model_diagnostics.infeasible import *
from idaes.core.util.model_diagnostics import DiagnosticsToolbox
import idaes.core.util.scaling as iscale

In [2]:
mass_flow_water = 0.995 * pyunits.kg / pyunits.s

salt_concentration = 5 * pyunits.g / pyunits.L
density = 995 * pyunits.kg / pyunits.m**3
mass_flow_salt = pyunits.convert(salt_concentration * mass_flow_water / density, to_units=pyunits.kg / pyunits.s)

# Pump parameters
pump_efficiency = 0.85 * pyunits.dimensionless
operating_pressure = 20 * pyunits.bar

A_comp = 4.2e-12 * pyunits.m/(pyunits.s * pyunits.Pa)               # membrane water permeability coefficient [m/s-Pa]
B_comp = 3e-8 * pyunits.m/(pyunits.s)                              # membrane salt permeability coefficient [m/s]
membrane_area = 50  * pyunits.m**2
atmospheric = 101325  * pyunits.Pa
deltaP = -3 * pyunits.bar
channel_height = 1 * pyunits.mm 
spacer_porosity = 0.75  * pyunits.dimensionless
RR = 0.45   * pyunits.dimensionless

mass_flow_salt()

0.004999999999999998

In [3]:
m = ConcreteModel()
m.fs = FlowsheetBlock(dynamic=False)
m.fs.ro_properties = NaClParameterBlock()

m.fs.RO = ReverseOsmosis0D(
    property_package=m.fs.ro_properties,
    has_pressure_change=True,
    pressure_change_type=PressureChangeType.calculated,
    mass_transfer_coefficient=MassTransferCoefficient.calculated,
    concentration_polarization_type=ConcentrationPolarizationType.calculated,
    module_type="spiral_wound",
)

m.fs.RO.inlet.pressure.fix(operating_pressure)
m.fs.RO.inlet.temperature.fix(298)
m.fs.RO.inlet.flow_mass_phase_comp[0, "Liq","H2O"].fix(mass_flow_water)
m.fs.RO.inlet.flow_mass_phase_comp[0, "Liq","NaCl"].fix(mass_flow_salt)    

# Fix (2) membrane properties
m.fs.RO.A_comp.fix(A_comp)
m.fs.RO.B_comp.fix(B_comp)

# Fix (4) module specifications
m.fs.RO.feed_side.channel_height.fix(channel_height)
m.fs.RO.feed_side.spacer_porosity.fix(spacer_porosity)
m.fs.RO.area.fix(membrane_area)
m.fs.RO.deltaP.fix(deltaP)                           
# m.fs.RO.recovery_vol_phase[0.0, "Liq"].fix(RR)

# (1) outlet state variable
m.fs.RO.permeate.pressure[0].fix(atmospheric)

TransformationFactory("network.expand_arcs").apply_to(m)  

In [4]:
# Set scaling factors
m.fs.ro_properties.set_default_scaling("flow_mass_phase_comp", 10, index=("Liq", "H2O"))
m.fs.ro_properties.set_default_scaling("flow_mass_phase_comp", 1e4, index=("Liq", "NaCl"))

iscale.constraint_scaling_transform(m.fs.RO.eq_recovery_vol_phase[0.0], 1e-2)
set_scaling_factor(m.fs.RO.area, 1e-3)

# Deal with low flows
m.fs.RO.feed_side.cp_modulus.setub(10)
m.fs.RO.feed_side.cp_modulus.setlb(0.001)

m.fs.RO.flux_mass_phase_comp.setub(0.1)
m.fs.RO.flux_mass_phase_comp.setlb(0)

m.fs.RO.feed_side.friction_factor_darcy.setub(200)

m.fs.RO.feed_side.K.setlb(1e-9)
# iscale.constraint_scaling_transform(m.fs.RO.feed_side.eq_dh, 1e-6)

# m.fs.RO.feed_side.N_Re[0.0,1.0].setlb(0.01)

# Release constraints related to low concentrations
# for item in [m.fs.RO.permeate_side, m.fs.RO.feed_side.properties_interface]:
#     for idx, param in item.items():
#         param.molality_phase_comp["Liq", "NaCl"].setlb(None)   ## None maybe...
#         param.pressure_osm_phase["Liq"].setlb(None)


# m.fs.RO.permeate_side[0.0,0.0].conc_mass_phase_comp["Liq","NaCl"].setlb(1e-6)

# for e in m.fs.RO.feed_side.eq_friction_factor:
#     iscale.constraint_scaling_transform(
#                 m.fs.RO.feed_side.eq_friction_factor[e], 1e-2
#             )
# for e in m.fs.RO.feed_side.eq_dP_dx:
#     iscale.constraint_scaling_transform(m.fs.RO.feed_side.eq_dP_dx[e], 1e-2)

calculate_scaling_factors(m)
print(f"dof = {degrees_of_freedom(m)}")

dof = 0


In [5]:
m.fs.RO.initialize()
solver = get_solver()
results = solver.solve(m)
assert_optimal_termination(results)

print("Estimated area: ", value(m.fs.RO.area), pyunits.get_units(m.fs.RO.area))
print("Estimated recovery: ", value(m.fs.RO.recovery_vol_phase[0.0, 'Liq']), pyunits.get_units(m.fs.RO.recovery_vol_phase[0.0, 'Liq']))
print("Estimated inlet pressure: ", value(m.fs.RO.inlet.pressure[0]), pyunits.get_units(m.fs.RO.inlet.pressure[0]))

m.fs.RO.flux_mass_phase_comp.display()

2026-03-09 11:09:02 [INFO] idaes.init.fs.RO.feed_side: Initialization Complete
2026-03-09 11:09:02 [INFO] idaes.init.fs.RO: Initialization Complete: optimal - Optimal Solution Found
Estimated area:  50.0 m**2
Estimated recovery:  0.2636904526578166 dimensionless
Estimated inlet pressure:  2000000.0 Pa
flux_mass_phase_comp : Mass flux across membrane at inlet and outlet
    Size=4, Index=fs._time*fs.RO.feed_side.length_domain*fs.ro_properties.phase_list*fs.ro_properties.component_list, Units=kg/m**2/s
    Key                       : Lower : Value                  : Upper : Fixed : Stale : Domain
     (0.0, 0.0, 'Liq', 'H2O') :     0 :    0.00617710250869535 :   0.1 : False : False :  Reals
    (0.0, 0.0, 'Liq', 'NaCl') :     0 :  1.637654694438384e-07 :   0.1 : False : False :  Reals
     (0.0, 1.0, 'Liq', 'H2O') :     0 :   0.004330505420099258 :   0.1 : False : False :  Reals
    (0.0, 1.0, 'Liq', 'NaCl') :     0 : 2.1705016672341852e-07 :   0.1 : False : False :  Reals


In [6]:
m.fs.RO.A_comp.fix(2.027e-11)
# m.fs.RO.inlet.pressure.unfix()
# m.fs.RO.recovery_vol_phase[0.0, "Liq"].fix(0.6)

In [7]:
solver = get_solver()
results = solver.solve(m)
assert_optimal_termination(results)

print("Estimated area: ", value(m.fs.RO.area), pyunits.get_units(m.fs.RO.area))
print("Estimated recovery: ", value(m.fs.RO.recovery_vol_phase[0.0, 'Liq']), pyunits.get_units(m.fs.RO.recovery_vol_phase[0.0, 'Liq']))
print("Estimated inlet pressure: ", value(m.fs.RO.inlet.pressure[0]), pyunits.get_units(m.fs.RO.inlet.pressure[0]))
print("A_comp: ", value(m.fs.RO.A_comp[0, 'H2O']), pyunits.get_units(m.fs.RO.A_comp[0, 'H2O']))

m.fs.RO.flux_mass_phase_comp.display()

Estimated area:  50.0 m**2
Estimated recovery:  0.7269671211173601 dimensionless
Estimated inlet pressure:  2000000.0 Pa
A_comp:  2.027e-11 m**2*s/kg
flux_mass_phase_comp : Mass flux across membrane at inlet and outlet
    Size=4, Index=fs._time*fs.RO.feed_side.length_domain*fs.ro_properties.phase_list*fs.ro_properties.component_list, Units=kg/m**2/s
    Key                       : Lower : Value                 : Upper : Fixed : Stale : Domain
     (0.0, 0.0, 'Liq', 'H2O') :     0 :    0.0267885054831199 :   0.1 : False : False :  Reals
    (0.0, 0.0, 'Liq', 'NaCl') :     0 :  2.20674773509246e-07 :   0.1 : False : False :  Reals
     (0.0, 1.0, 'Liq', 'H2O') :     0 : 0.0021799384957862057 :   0.1 : False : False :  Reals
    (0.0, 1.0, 'Liq', 'NaCl') :     0 : 5.668392204504228e-07 :   0.1 : False : False :  Reals


In [8]:
flow_vol = 500 * pyunits.m**3/pyunits.hour
density = 995 * pyunits.kg/pyunits.m**3

feed_conc = 35 * pyunits.g/pyunits.liter
membrane_area = 20000 * pyunits.m**2

mass_flow_water = pyunits.convert(flow_vol * density, to_units=pyunits.kg/pyunits.s) 
mass_flow_salt = pyunits.convert(feed_conc, to_units=pyunits.kg/pyunits.m**3) * pyunits.convert(flow_vol, to_units=pyunits.m**3/pyunits.s)

print("Mass flow of water: ", value(mass_flow_water), pyunits.get_units(mass_flow_water))
print("Mass flow of salt: ", value(mass_flow_salt), pyunits.get_units(mass_flow_salt))

Mass flow of water:  138.19444444444446 kg/s
Mass flow of salt:  4.861111111111111 kg/s


In [9]:
m.fs.RO.inlet.flow_mass_phase_comp[0, "Liq","H2O"].fix(mass_flow_water)
m.fs.RO.inlet.flow_mass_phase_comp[0, "Liq","NaCl"].fix(mass_flow_salt)

m.fs.RO.area.fix(membrane_area)

m.fs.ro_properties.set_default_scaling("flow_mass_phase_comp", 1e-2, index=("Liq", "H2O"))
m.fs.ro_properties.set_default_scaling("flow_mass_phase_comp", 1e1, index=("Liq", "NaCl"))
set_scaling_factor(m.fs.RO.area, 1e-5)

# m.fs.ro_properties.set_default_scaling("pressure", 1e-8, index=("Liq",))
calculate_scaling_factors(m)
degrees_of_freedom(m)

0

In [10]:
m.fs.RO.inlet.pressure.unfix()

In [11]:
solver = get_solver()
results = solver.solve(m)
assert_optimal_termination(results)

print("Estimated area: ", value(m.fs.RO.area), pyunits.get_units(m.fs.RO.area))
print("Estimated recovery: ", value(m.fs.RO.recovery_vol_phase[0.0, 'Liq']), pyunits.get_units(m.fs.RO.recovery_vol_phase[0.0, 'Liq']))
print("Estimated inlet pressure: ", value(m.fs.RO.inlet.pressure[0]), pyunits.get_units(m.fs.RO.inlet.pressure[0]))

m.fs.RO.flux_mass_phase_comp.display()

Estimated area:  20000.0 m**2
Estimated recovery:  0.5872535659942135 dimensionless
Estimated inlet pressure:  3701727.964408293 Pa
flux_mass_phase_comp : Mass flux across membrane at inlet and outlet
    Size=4, Index=fs._time*fs.RO.feed_side.length_domain*fs.ro_properties.phase_list*fs.ro_properties.component_list, Units=kg/m**2/s
    Key                       : Lower : Value                  : Upper : Fixed : Stale : Domain
     (0.0, 0.0, 'Liq', 'H2O') :     0 :   0.008164516960019118 :   0.1 : False : False :  Reals
    (0.0, 0.0, 'Liq', 'NaCl') :     0 : 1.1987478343304713e-06 :   0.1 : False : False :  Reals
     (0.0, 1.0, 'Liq', 'H2O') :     0 :  2.446930179836273e-05 :   0.1 : False : False :  Reals
    (0.0, 1.0, 'Liq', 'NaCl') :     0 : 1.1357766030123996e-06 :   0.1 : False : False :  Reals


In [12]:
m.fs.RO.inlet.pressure.unfix()
m.fs.RO.recovery_vol_phase[0.0, "Liq"].fix(0.9)

In [13]:
solver = get_solver()
results = solver.solve(m)
assert_optimal_termination(results)

print("Estimated area: ", value(m.fs.RO.area), pyunits.get_units(m.fs.RO.area))
print("Estimated recovery: ", value(m.fs.RO.recovery_vol_phase[0.0, 'Liq']), pyunits.get_units(m.fs.RO.recovery_vol_phase[0.0, 'Liq']))
print("Estimated inlet pressure: ", value(m.fs.RO.inlet.pressure[0]), pyunits.get_units(m.fs.RO.inlet.pressure[0]))
print("Estimated pressure drop: ", pyunits.convert(m.fs.RO.deltaP[0], to_units=pyunits.bar)() )

m.fs.RO.flux_mass_phase_comp.display()

Estimated area:  20000.0 m**2
Estimated recovery:  0.9 dimensionless
Estimated inlet pressure:  4164490.860731401 Pa
Estimated pressure drop:  -3.0000000000000004
flux_mass_phase_comp : Mass flux across membrane at inlet and outlet
    Size=4, Index=fs._time*fs.RO.feed_side.length_domain*fs.ro_properties.phase_list*fs.ro_properties.component_list, Units=kg/m**2/s
    Key                       : Lower : Value                  : Upper : Fixed : Stale : Domain
     (0.0, 0.0, 'Liq', 'H2O') :     0 :   0.012549001850089286 :   0.1 : False : False :  Reals
    (0.0, 0.0, 'Liq', 'NaCl') :     0 : 1.2881958483340236e-06 :   0.1 : False : False :  Reals
     (0.0, 1.0, 'Liq', 'H2O') :     0 : 1.5129243124843712e-06 :   0.1 : False : False :  Reals
    (0.0, 1.0, 'Liq', 'NaCl') :     0 :  5.409278437236311e-07 :   0.1 : False : False :  Reals


In [14]:
print_variables_close_to_bounds(m)

In [15]:
print_infeasible_constraints(m)

In [16]:
# for d in list(m.fs.RO.length_domain):
#     m.fs.RO.feed_side.velocity[0, d].setub(1)
#     m.fs.RO.feed_side.velocity[0, d].setlb(0.01)

# Deal with low flows
# m.fs.RO.feed_side.cp_modulus.setub(5)
# m.fs.RO.feed_side.cp_modulus.setlb(0.01)

# Deals with low feed salinity
# for e in m.fs.RO.flux_mass_phase_comp:
#     if e[-1] == "NaCl":
#         m.fs.RO.flux_mass_phase_comp[e].setlb(1e-9)
#         m.fs.RO.flux_mass_phase_comp[e].setub(1e-1)

# iscale.constraint_scaling_transform(
#             m.fs.RO.mixed_permeate[0.0].eq_flow_vol_phase["Liq"], 1e-6
# #         )
# iscale.constraint_scaling_transform(m.fs.RO.eq_mass_frac_permeate[0.0,0.0,'NaCl'], 1e-6)
# iscale.constraint_scaling_transform(m.fs.RO.eq_mass_frac_permeate[0.0,1.0,'NaCl'], 1e-6)
# iscale.constraint_scaling_transform(m.fs.RO.eq_recovery_vol_phase[0.0], 1e-2)
# m.fs.RO.flux_mass_phase_comp.setub(1)
# m.fs.RO.flux_mass_phase_comp.setlb(0)

# m.fs.RO.feed_side.K.setlb(1e-6)
# m.fs.RO.feed_side.friction_factor_darcy.setub(200)
# iscale.constraint_scaling_transform(m.fs.RO.feed_side.eq_dh, 1e-6)

In [17]:
RR = 0.9
membrane_area = 20000 * pyunits.m**2
mass_flow_water = (1000*500/3600) * pyunits.kg / pyunits.s                                           # 500 m3/h  (0.13 Kg/s)
Concentration = 4                                                                                    # 4 g/l
Concentration_ppm = Concentration * 1000                                                             # 4000 ppm
Density = 1000                                                                                       # 1000 kg/m3
mass_flow_salt = mass_flow_water * Concentration_ppm / (1000000)    
                                 # Mass flow rate of salt kg/s  / Divided by a million as the concentration is used in ppm                                                                                      #
A_comp = (7.2994/(1000*3600*100000)) * pyunits.m/(pyunits.s * pyunits.Pa)                            # 2.027 E-11 membrane water permeability coefficient [m/s-Pa] - A = 7.2994    L/m2 * h * bar